# QuantumHive — Pipeline Completo Kaggle GPU

**Qué hace este notebook:**
1. Carga tus CSVs multitemporales de MT5 (subidos como dataset)
2. Genera `datatb_fusion.parquet` (indicadores + confluencias + targets)
3. Entrena Bot Madre (RecurrentPPO) + 3 hijos (PPO)
4. Exporta 4 modelos `.onnx` para MT5
5. Guarda reporte de precisión

**Hardware:** GPU T4 (gratis en Kaggle, 30h/semana)

In [ ]:
# Celda 1: Instalar dependencias
!pip install -q stable-baselines3[extra] sb3-contrib gymnasium onnx onnxruntime pandas numpy pyarrow fastparquet
print("OK dependencias")

In [ ]:
# Celda 2: Imports y paths
from __future__ import annotations
import os, sys, json, warnings
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import torch
from stable_baselines3 import PPO
from sb3_contrib import RecurrentPPO
from stable_baselines3.common.vec_env import DummyVecEnv
warnings.filterwarnings("ignore")

# Paths Kaggle
INPUT = Path("/kaggle/input/quantumhive-csvs")  # <- nombre de tu dataset
WORKING = Path("/kaggle/working")
MODELS = WORKING / "modelos_onnx"
MODELS.mkdir(exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Dispositivo: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    torch.backends.cudnn.benchmark = True

---
## FASE A: Generar datatb_fusion desde CSVs MT5

In [ ]:
# Celda 3: Leer CSVs multitemporales
def leer_csv_mt5(nombre: str) -> pd.DataFrame | None:
    ruta = INPUT / nombre
    if not ruta.exists():
        print(f"[SKIP] No existe: {nombre}")
        return None
    df = pd.read_csv(ruta, sep='\t', engine='python')
    df.columns = [c.strip().strip('<>') for c in df.columns]
    df['datetime'] = pd.to_datetime(df['DATE'] + ' ' + df['TIME'], format='%Y.%m.%d %H:%M:%S', errors='coerce')
    for col in ['OPEN','HIGH','LOW','CLOSE','TICKVOL','VOL','SPREAD']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col].astype(str).str.replace(',','.'), errors='coerce')
    return df.dropna(subset=['datetime','CLOSE']).set_index('datetime').sort_index()

m1 = leer_csv_mt5('datatb.csv')
tf_data = {
    'M5': leer_csv_mt5('datatb_m5.csv'),
    'M15': leer_csv_mt5('datatb_m15.csv'),
    'H1': leer_csv_mt5('datatb_H1.csv'),
    'H4': leer_csv_mt5('datatb_H4.csv'),
    'D1': leer_csv_mt5('datatb_daily.csv'),
}
print(f"\nM1 cargado: {len(m1):,} filas")
for k,v in tf_data.items():
    if v is not None:
        print(f"{k}: {len(v):,} filas")

In [ ]:
# Celda 4: Indicadores técnicos en M1
def calcular_indicadores(df: pd.DataFrame) -> pd.DataFrame:
    d = df.copy()
    close, high, low = d['CLOSE'], d['HIGH'], d['LOW']
    vol = d.get('TICKVOL', d.get('VOL', pd.Series(1, index=d.index)))
    
    # RSI 14
    delta = close.diff()
    gain = delta.where(delta>0,0).rolling(14).mean()
    loss = (-delta.where(delta<0,0)).rolling(14).mean()
    rs = gain/loss
    d['rsi'] = 100 - (100/(1+rs))
    
    # EMAs
    d['ema_fast'] = close.ewm(span=12, adjust=False).mean()
    d['ema_slow'] = close.ewm(span=26, adjust=False).mean()
    d['ema_200'] = close.ewm(span=200, adjust=False).mean()
    
    # Bollinger + BB% + BBW
    sma20 = close.rolling(20).mean()
    std20 = close.rolling(20).std()
    d['bb_upper'] = sma20 + 2*std20
    d['bb_mid'] = sma20
    d['bb_lower'] = sma20 - 2*std20
    bw = d['bb_upper'] - d['bb_lower']
    d['bb_pct_b'] = (close - d['bb_lower']) / bw
    d['bbw'] = bw / d['bb_mid']
    
    # ATR 14
    tr1, tr2, tr3 = high-low, abs(high-close.shift()), abs(low-close.shift())
    d['atr'] = pd.concat([tr1,tr2,tr3],axis=1).max(axis=1).rolling(14).mean()
    
    # ADX (Wilder simplificado)
    plus_dm = high.diff().where(high.diff()>low.diff(),0)
    minus_dm = (-low.diff()).where(low.diff()>high.diff(),0)
    atr_smooth = pd.concat([tr1,tr2,tr3],axis=1).max(axis=1).ewm(alpha=1/14,adjust=False).mean()
    plus_di = 100*plus_dm.ewm(alpha=1/14,adjust=False).mean()/atr_smooth
    minus_di = 100*minus_dm.ewm(alpha=1/14,adjust=False).mean()/atr_smooth
    dx = 100*abs(plus_di-minus_di)/(plus_di+minus_di)
    d['adx'] = dx.ewm(alpha=1/14,adjust=False).mean()
    d['plus_di'], d['minus_di'] = plus_di, minus_di
    
    # MACD
    ema12, ema26 = close.ewm(span=12,adjust=False).mean(), close.ewm(span=26,adjust=False).mean()
    d['macd'] = ema12-ema26
    d['macd_signal'] = d['macd'].ewm(span=9,adjust=False).mean()
    d['macd_hist'] = d['macd'] - d['macd_signal']
    
    # Volumen
    d['volume_spike'] = vol / vol.ewm(span=20,adjust=False).mean()
    return d

m1 = calcular_indicadores(m1)
print(f"Indicadores OK. NaN: {m1.isna().mean().mean()*100:.2f}%")

In [ ]:
# Celda 5: Alinear timeframes superiores a M1
df = m1.reset_index()
for tf_name, tf_df in tf_data.items():
    if tf_df is None or tf_df.empty: continue
    cols = tf_df[['OPEN','HIGH','LOW','CLOSE','SPREAD']].copy()
    cols.columns = [f"{c.lower()}_{tf_name.lower()}" for c in cols.columns]
    cols = cols.reset_index()
    df = pd.merge_asof(df.sort_values('datetime'), cols.sort_values('datetime'),
                       on='datetime', direction='backward')
df = df.set_index('datetime').sort_index()
print(f"Fusion OK: {len(df)} filas x {len(df.columns)} columnas")

In [ ]:
# Celda 6: Confluencias y Targets para RL
# EMA trend direction por TF
for tf in ['m1','m5','m15','h1','h4','daily']:
    col = f'close_{tf}'
    if col in df.columns:
        ema = df[col].ewm(span=20,adjust=False).mean()
        df[f'ema_trend_{tf}'] = np.where(df[col]>ema*1.0002, 1, np.where(df[col]<ema*0.9998,-1,0))

# RSI signal
df['rsi_signal'] = np.where(df['rsi']>70, -1, np.where(df['rsi']<30, 1, 0))

# BB position
df['bb_position'] = np.where(df['CLOSE']<=df['bb_lower']*1.0001, -1,
                              np.where(df['CLOSE']>=df['bb_upper']*0.9999, 1, 0))

# MACD confluence
df['macd_confluence'] = np.where((df['macd']>df['macd_signal'])&(df['macd_hist']>df['macd_hist'].shift(1)),1,
                                  np.where((df['macd']<df['macd_signal'])&(df['macd_hist']<df['macd_hist'].shift(1)),-1,0))

# Targets (horizon 60 min)
future = df['CLOSE'].shift(-60)
df['return_future'] = (future - df['CLOSE'])/df['CLOSE']
thr = 0.001
df['target_direction'] = np.where(df['return_future']>thr, 2,
                                   np.where(df['return_future']<-thr, 0, 1))
df['target_reward'] = df['return_future'] / (df['atr']/df['CLOSE'])

# Limpiar warmup (EMA200)
min_idx = df['ema_200'].first_valid_index()
df = df.loc[min_idx:].dropna(subset=['target_direction'])
print(f"Dataset final: {len(df):,} filas | Target LONG={(df['target_direction']==2).mean()*100:.1f}% | FLAT={(df['target_direction']==1).mean()*100:.1f}% | SHORT={(df['target_direction']==0).mean()*100:.1f}%")

In [ ]:
# Celda 7: Guardar datatb_fusion
ruta_parquet = WORKING / 'datatb_fusion.parquet'
df.to_parquet(ruta_parquet, compression='zstd')
print(f"Guardado: {ruta_parquet} ({ruta_parquet.stat().st_size/1024/1024:.1f} MB)")

# Descargar archivo para verificar
from IPython.display import FileLink
FileLink(ruta_parquet)

---
## FASE B: Entrenamiento RL (GPU T4)

In [ ]:
# Celda 8: Entornos simplificados para RL
import gymnasium as gym
from typing import Any

class EntornoMadreKaggle(gym.Env):
    metadata = {'render_modes': []}
    def __init__(self, df: pd.DataFrame, features: list[str], horizon: int = 60):
        super().__init__()
        self.df = df.reset_index(drop=True)
        self.features = [f for f in features if f in df.columns]
        self.horizon = horizon
        self.window = 200
        self.action_space = gym.spaces.Discrete(3)
        self.observation_space = gym.spaces.Box(low=-np.inf, high=np.inf, shape=(len(self.features)+3,), dtype=np.float32)
        self._idx = self.window
        self._last_pos = 1  # FLAT
    def reset(self, seed=None, options=None):
        self._idx = self.window
        self._last_pos = 1
        return self._obs(), {}
    def _obs(self):
        row = self.df.iloc[self._idx]
        obs = list(row[self.features].fillna(0).values)
        obs += [self._last_pos, row.get('bb_pct_b',0.5), row.get('bbw',0.05)]
        return np.array(obs, dtype=np.float32)
    def step(self, action):
        row = self.df.iloc[self._idx]
        future_price = self.df.iloc[min(self._idx+self.horizon, len(self.df)-1)]['CLOSE']
        current_price = row['CLOSE']
        ret = (future_price - current_price)/current_price
        
        # Reward shaping
        if action == 0:  # SHORT
            reward = -ret * 100 - abs(ret)*0.5
        elif action == 2:  # LONG
            reward = ret * 100 - abs(ret)*0.5
        else:  # WAIT
            reward = -abs(ret)*10  # Penalidad por no operar si hay movimiento
        
        self._last_pos = action
        self._idx += 1
        terminated = self._idx >= len(self.df) - self.horizon - 1
        truncated = False
        return self._obs(), float(reward), terminated, truncated, {'return': ret}

print("Entorno definido OK")

In [ ]:
# Celda 9: Features para Bot Madre (14 + confluencias)
FEATURES_MADRE = [
    'CLOSE','HIGH','LOW','OPEN','volume_spike',
    'rsi','ema_fast','ema_slow','bb_upper','bb_lower',
    'atr','adx','macd','macd_signal',
    'ema_trend_m1','ema_trend_m5','ema_trend_h1','macd_confluence','bb_position'
]

def evaluar_modelo(model, env, n_steps=3000):
    obs, _ = env.reset()
    rewards, rets, wins, losses = [], [], 0, 0
    for _ in range(n_steps):
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, done, trunc, info = env.step(int(action))
        rewards.append(reward)
        ret = info.get('return',0)
        rets.append(ret)
        if ret > 0.001: wins += 1
        elif ret < -0.001: losses += 1
        if done or trunc: break
    wr = wins/(wins+losses) if (wins+losses)>0 else 0
    return {
        'mean_reward': float(np.mean(rewards)),
        'winrate': round(wr, 4),
        'profit_factor': round(abs(sum(r for r in rets if r>0))/abs(sum(r for r in rets if r<0)), 2) if any(r<0 for r in rets) else 999,
        'trades': wins+losses,
        'total_return_pct': round(sum(rets)*100, 3)
    }

print("Helpers OK")

In [ ]:
# Celda 10: Entrenar BOT MADRE (RecurrentPPO)
print("\n=== BOT MADRE ===")
env_madre = DummyVecEnv([lambda: EntornoMadreKaggle(df, FEATURES_MADRE)])

model_madre = RecurrentPPO(
    'MlpLstmPolicy',
    env_madre,
    verbose=1,
    device=DEVICE,
    learning_rate=3e-4,
    n_steps=2048,
    batch_size=256,
    n_epochs=10,
    gamma=0.995,
    gae_lambda=0.95
)
model_madre.learn(total_timesteps=300_000)

env_test = EntornoMadreKaggle(df, FEATURES_MADRE)
stats_madre = evaluar_modelo(model_madre, env_test, n_steps=5000)
print(f"MADRE: {stats_madre}")

model_madre.save(str(MODELS / 'bot_madre_ppo'))
print("Madre guardado.")

In [ ]:
# Celda 11: Entrenar BOT REVERSIÓN (filtra donde RSI sobrecompra/sobrevende)
print("\n=== BOT REVERSIÓN ===")
df_rev = df[(df['rsi']>70) | (df['rsi']<30)].copy()
env_rev = DummyVecEnv([lambda: EntornoMadreKaggle(df_rev, FEATURES_MADRE)])

model_rev = PPO('MlpPolicy', env_rev, verbose=1, device=DEVICE,
                learning_rate=3e-4, n_steps=1024, batch_size=128, n_epochs=10)
model_rev.learn(total_timesteps=200_000)

stats_rev = evaluar_modelo(model_rev, EntornoMadreKaggle(df_rev, FEATURES_MADRE), n_steps=3000)
print(f"REVERSIÓN: {stats_rev}")
model_rev.save(str(MODELS / 'bot_reversion_ppo'))

In [ ]:
# Celda 12: Entrenar BOT CONTINUACIÓN (filtra donde ADX>25 = tendencia fuerte)
print("\n=== BOT CONTINUACIÓN ===")
df_cont = df[df['adx'] > 25].copy()
env_cont = DummyVecEnv([lambda: EntornoMadreKaggle(df_cont, FEATURES_MADRE)])

model_cont = PPO('MlpPolicy', env_cont, verbose=1, device=DEVICE,
                 learning_rate=3e-4, n_steps=1024, batch_size=128, n_epochs=10)
model_cont.learn(total_timesteps=200_000)

stats_cont = evaluar_modelo(model_cont, EntornoMadreKaggle(df_cont, FEATURES_MADRE), n_steps=3000)
print(f"CONTINUACIÓN: {stats_cont}")
model_cont.save(str(MODELS / 'bot_continuacion_ppo'))

In [ ]:
# Celda 13: Entrenar BOT SCALPER (BB% extremos + volumen)
print("\n=== BOT SCALPER ===")
df_scalp = df[(df['bb_pct_b']<0.05) | (df['bb_pct_b']>0.95)].copy()
env_scalp = DummyVecEnv([lambda: EntornoMadreKaggle(df_scalp, FEATURES_MADRE)])

model_scalp = PPO('MlpPolicy', env_scalp, verbose=1, device=DEVICE,
                  learning_rate=3e-4, n_steps=1024, batch_size=128, n_epochs=10)
model_scalp.learn(total_timesteps=200_000)

stats_scalp = evaluar_modelo(model_scalp, EntornoMadreKaggle(df_scalp, FEATURES_MADRE), n_steps=3000)
print(f"SCALPER: {stats_scalp}")
model_scalp.save(str(MODELS / 'bot_scalper_ppo'))

---
## FASE C: Exportar ONNX para MT5

In [ ]:
# Celda 14: Exportar a ONNX (opset 12 compatible MT5)
import onnx
from stable_baselines3.common.policies import BasePolicy

def export_ppo_onnx(model, env, path: str, opset=12):
    obs, _ = env.reset()
    obs_t = torch.as_tensor(obs, dtype=torch.float32).unsqueeze(0)
    if DEVICE=='cuda':
        obs_t = obs_t.cuda()
    
    # Para RecurrentPPO necesitamos lstm_states
    try:
        lstm_states = None
        episode_starts = torch.ones((1,), dtype=torch.float32)
        if hasattr(model.policy, 'lstm_actor'):
            # RecurrentPPO: inferir shape de hidden states
            dummy = model.policy(obs_t, lstm_states, episode_starts)
            # Exportar solo el actor MLP (no LSTM) para simplificar MT5
            # O usar torch.jit.trace con cuidado
            actor = model.policy.mlp_extractor
            traced = torch.jit.trace(actor, obs_t)
            torch.onnx.export(traced, obs_t, path,
                              export_params=True, opset_version=opset,
                              input_names=['obs'], output_names=['action_logits'])
        else:
            # PPO standard
            traced = torch.jit.trace(model.policy, obs_t)
            torch.onnx.export(traced, obs_t, path,
                              export_params=True, opset_version=opset,
                              input_names=['obs'], output_names=['action_logits'])
    except Exception as e:
        # Fallback: exportar solo el extractor de features
        mlp = model.policy.mlp_extractor
        torch.onnx.export(mlp, obs_t, path,
                          export_params=True, opset_version=opset,
                          input_names=['obs'], output_names=['features'])
        print(f"WARN: export alternativo para {path}: {e}")
    print(f"ONNX exportado: {path}")
    return path

# Exportar 4 modelos
env_dummy = EntornoMadreKaggle(df.head(1000), FEATURES_MADRE)
export_ppo_onnx(model_madre, env_dummy, str(MODELS / 'bot_madre.onnx'))
export_ppo_onnx(model_rev, env_dummy, str(MODELS / 'bot_reversion.onnx'))
export_ppo_onnx(model_cont, env_dummy, str(MODELS / 'bot_continuacion.onnx'))
export_ppo_onnx(model_scalp, env_dummy, str(MODELS / 'bot_scalper.onnx'))

# Verificar ONNX
for f in MODELS.glob('*.onnx'):
    m = onnx.load(f)
    onnx.checker.check_model(m)
    print(f"✓ {f.name}: OK ({f.stat().st_size/1024:.0f} KB)")

In [ ]:
# Celda 15: Reporte final + links de descarga
reporte = {
    'timestamp': datetime.now(timezone.utc).isoformat(),
    'device': DEVICE,
    'dataset_rows': len(df),
    'bots': {
        'madre': stats_madre,
        'reversion': stats_rev,
        'continuacion': stats_cont,
        'scalper': stats_scalp,
    },
    'archivos': [str(f.name) for f in MODELS.iterdir()]
}

ruta_reporte = WORKING / 'reporte_kaggle.json'
with open(ruta_reporte, 'w') as f:
    json.dump(reporte, f, indent=2)

print("\n" + "="*60)
print("RESULTADOS FINALES")
print("="*60)
for nombre, st in reporte['bots'].items():
    wr = st.get('winrate',0)
    pf = st.get('profit_factor',0)
    mr = st.get('mean_reward',0)
    print(f"  {nombre:12s} | WR={wr:.2%} | PF={pf} | MR={mr:.4f}")
print("="*60)

# Links de descarga
print("\nArchivos listos en /kaggle/working/:")
for f in sorted(MODELS.glob('*')) + [ruta_reporte, WORKING/'datatb_fusion.parquet']:
    if f.exists():
        print(f"  {f.name}")
        display(FileLink(f))